In [1]:
# %%
# Get the current working directory
import os 
current_dir = os.getcwd()
print("Current working directory:", current_dir)

import pandas as pd
import scanpy as sc
import anndata as ad
from tqdm import tqdm
import matplotlib.pyplot as plt # import matplotlib to visualize our qc metrics
import subprocess
import sys
import seaborn as sns
import numpy as np
import cupy as cp
import cudf
import scipy.sparse as sp

# magic incantation to help matplotlib work with our jupyter notebook
%matplotlib inline 

sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white')

sns.set(style="whitegrid")

# %%
# Directory containing .h5ad files
adata_dir = "/gpfs/commons/datasets/controlled/CZI/tabula-sapiens/TabulaSapiens_v2/GeneExpressionMatrices"

# List all .h5ad files
adata_files = [os.path.join(adata_dir, f) for f in os.listdir(adata_dir) if f.endswith(".h5ad")]

# Collect all the data into a list 
adata_list = []

# Load all the .h5ad files
for adata_file in tqdm(adata_files):
    print(f"Reading {adata_file}")
    adata = sc.read_h5ad(adata_file)
    print(f"Number of cells and genes: {adata.n_obs}, {adata.n_vars}")
    adata_list.append(adata)

print(f"Loaded {len(adata_list)} datasets")

# %%
# Let's first extract the adata.obs from each one and save those into one object
obs_list = [adata.obs for adata in adata_list]
obs = pd.concat(obs_list)
obs.head()

# %%
# SANITY CHECK 1: Count total cells and SS2 cells before processing
print("=== INITIAL SANITY CHECKS ===")
total_cells_initial = sum(adata.n_obs for adata in adata_list)
ss2_cells_initial = sum((adata.obs["assay"] == "SS2").sum() for adata in adata_list)
print(f"Total cells across all datasets: {total_cells_initial:,}")
print(f"Total SS2 cells across all datasets: {ss2_cells_initial:,}")

# Check assay types distribution
assay_counts = pd.concat([adata.obs["assay"] for adata in adata_list]).value_counts()
print(f"Assay type distribution:\n{assay_counts}")

# Step 1: Collect Gene Lists from All Datasets
gene_sets = [set(adata.var_names) for adata in adata_list]

# Find Common Genes Across All Datasets
common_genes = sorted(set.intersection(*gene_sets))  # Sorting ensures consistent order
print(f"Found {len(common_genes)} common genes across all datasets.")

# SANITY CHECK 2: Verify gene overlap
print("\n=== GENE OVERLAP CHECKS ===")
total_unique_genes = len(set.union(*gene_sets))
print(f"Total unique genes across all datasets: {total_unique_genes:,}")
print(f"Common genes: {len(common_genes):,}")
print(f"Genes lost due to intersection: {total_unique_genes - len(common_genes):,}")

# Check individual dataset gene counts
for i, (adata, gene_set) in enumerate(zip(adata_list, gene_sets)):
    genes_in_common = len(gene_set.intersection(set(common_genes)))
    print(f"Dataset {i}: {len(gene_set):,} genes, {genes_in_common:,} in common set")

# Step 2: Align All Datasets to the Common Gene Set & Keep Only `SS2`
X_list = []   # Sparse expression matrices
obs_list = [] # Cell metadata
var_final = None  # Store final gene metadata

# Track processing statistics
processing_stats = {
    'datasets_processed': 0,
    'datasets_skipped_no_ss2': 0,
    'datasets_skipped_no_raw_counts': 0,
    'total_ss2_cells_kept': 0,
    'cell_counts_per_dataset': []
}

for i, adata in enumerate(tqdm(adata_list)):
    print(f"\n--- Processing dataset {i} ---")
    
    # SANITY CHECK 3: Pre-processing checks
    initial_cells = adata.n_obs
    initial_genes = adata.n_vars
    print(f"Initial: {initial_cells:,} cells, {initial_genes:,} genes")
    
    # Subset only `SS2` cells
    ss2_cells = adata.obs["assay"] == "SS2"
    ss2_count = ss2_cells.sum()
    print(f"SS2 cells in this dataset: {ss2_count:,}")
    
    if ss2_count == 0:
        print("❌ Skipping: No SS2 cells")
        processing_stats['datasets_skipped_no_ss2'] += 1
        continue

    adata = adata[ss2_cells, :].copy()  # Keep only SS2, make a copy to avoid view issues

    # Replace `adata.X` with `raw_counts`
    if "raw_counts" in adata.layers:
        print(f"✅ Using `adata.layers['raw_counts']` for {adata.shape[0]} cells")
        
        # SANITY CHECK 4: Verify raw_counts properties
        raw_counts = adata.layers["raw_counts"]
        print(f"Raw counts - Shape: {raw_counts.shape}, Type: {type(raw_counts)}")
        if sp.issparse(raw_counts):
            print(f"Raw counts - Sparse format: {raw_counts.format}, Non-zero: {raw_counts.nnz:,}")
        
        # Check for negative values (shouldn't exist in raw counts)
        if sp.issparse(raw_counts):
            min_val = raw_counts.min()
        else:
            min_val = raw_counts.min()
        
        if min_val < 0:
            print(f"⚠️  WARNING: Found negative values in raw_counts (min: {min_val})")
        else:
            print(f"✅ Raw counts are non-negative (min: {min_val})")
        
        adata.X = adata.layers["raw_counts"].copy()  # Ensure raw counts are used
    else:
        print(f"❌ Skipping: `raw_counts` layer missing. Available layers: {list(adata.layers.keys())}")
        processing_stats['datasets_skipped_no_raw_counts'] += 1
        continue

    # SANITY CHECK 5: Verify gene subsetting
    genes_before_subset = adata.n_vars
    genes_missing = set(common_genes) - set(adata.var_names)
    if genes_missing:
        print(f"⚠️  WARNING: {len(genes_missing)} common genes missing from this dataset")
        print(f"Missing genes (first 5): {list(genes_missing)[:5]}")

    # Subset and reorder genes to match the common gene set
    adata = adata[:, list(common_genes)].copy()  # **Reorders genes correctly!**
    print(f"After gene subsetting: {adata.shape[0]} cells, {adata.shape[1]} genes")
    
    # SANITY CHECK 6: Verify gene order consistency
    if var_final is not None:
        if not all(adata.var_names == var_final.index):
            print("❌ ERROR: Gene order mismatch!")
            # Show first few mismatches
            mismatches = adata.var_names != var_final.index
            if mismatches.any():
                mismatch_idx = np.where(mismatches)[0][:5]
                print(f"First mismatches at positions: {mismatch_idx}")
                for idx in mismatch_idx:
                    print(f"  Position {idx}: '{adata.var_names[idx]}' vs '{var_final.index[idx]}'")
            raise ValueError("Gene order consistency check failed!")
        else:
            print("✅ Gene order consistent with previous datasets")

    # Ensure `X` is sparse
    if not sp.issparse(adata.X):
        print(f"Converting X to sparse (was {type(adata.X)})")
        adata.X = sp.csr_matrix(adata.X)
    else:
        print(f"✅ X is already sparse ({adata.X.format})")

    # SANITY CHECK 7: Expression matrix properties
    print(f"Expression matrix - Shape: {adata.X.shape}, Non-zero: {adata.X.nnz:,}")
    print(f"Sparsity: {100 * (1 - adata.X.nnz / (adata.X.shape[0] * adata.X.shape[1])):.2f}%")

    X_list.append(adata.X)  # Store aligned sparse matrix
    obs_list.append(adata.obs.copy())  # Store cell metadata

    # Use the first dataset's `var` as the final gene metadata
    if var_final is None:
        var_final = adata.var.loc[list(common_genes)].copy()  # Keep gene metadata
        print("✅ Set gene metadata from first dataset")

    # Update processing stats
    processing_stats['datasets_processed'] += 1
    processing_stats['total_ss2_cells_kept'] += adata.shape[0]
    processing_stats['cell_counts_per_dataset'].append(adata.shape[0])

# SANITY CHECK 8: Processing summary
print("\n=== PROCESSING SUMMARY ===")
print(f"Datasets loaded: {len(adata_list)}")
print(f"Datasets processed: {processing_stats['datasets_processed']}")
print(f"Datasets skipped (no SS2): {processing_stats['datasets_skipped_no_ss2']}")
print(f"Datasets skipped (no raw_counts): {processing_stats['datasets_skipped_no_raw_counts']}")
print(f"Total SS2 cells kept: {processing_stats['total_ss2_cells_kept']:,}")
print(f"Expected total from initial count: {ss2_cells_initial:,}")

if processing_stats['total_ss2_cells_kept'] != ss2_cells_initial:
    difference = ss2_cells_initial - processing_stats['total_ss2_cells_kept']
    print(f"⚠️  DIFFERENCE: {difference:,} cells lost during processing")
else:
    print("✅ Cell count matches initial SS2 count!")

# Step 3: Merge Sparse Matrices & Metadata
print("\n=== MERGING DATA ===")
print("Merging sparse matrices and metadata...")

# SANITY CHECK 9: Pre-merge verification
expected_total_cells = sum(X.shape[0] for X in X_list)
expected_total_obs = sum(len(obs) for obs in obs_list)
print(f"Expected cells from X matrices: {expected_total_cells:,}")
print(f"Expected cells from obs DataFrames: {expected_total_obs:,}")

if expected_total_cells != expected_total_obs:
    print("❌ ERROR: Mismatch between X matrices and obs DataFrames!")
    raise ValueError("X and obs dimension mismatch!")

X_merged = sp.vstack(X_list)  # Efficient sparse merging
obs_merged = pd.concat(obs_list, ignore_index=True)  # Faster `obs` concatenation

# SANITY CHECK 10: Post-merge verification
print(f"Merged X shape: {X_merged.shape}")
print(f"Merged obs shape: {obs_merged.shape}")
print(f"var shape: {var_final.shape}")

if X_merged.shape[0] != obs_merged.shape[0]:
    print("❌ ERROR: X and obs row count mismatch after merging!")
    raise ValueError("Merged X and obs dimension mismatch!")

if X_merged.shape[1] != var_final.shape[0]:
    print("❌ ERROR: X columns and var row count mismatch!")
    raise ValueError("Merged X and var dimension mismatch!")

print("✅ Merged dimensions are consistent!")

# Step 4: Reconstruct the Final AnnData Object with RAW Counts
merged_adata = ad.AnnData(X=X_merged, obs=obs_merged, var=var_final)

# SANITY CHECK 11: Final AnnData object verification
print("\n=== FINAL VERIFICATION ===")
print(f"Final AnnData shape: {merged_adata.shape}")
print(f"Final cell count: {merged_adata.n_obs:,}")
print(f"Final gene count: {merged_adata.n_vars:,}")

# Check if all cells are SS2
final_assay_counts = merged_adata.obs["assay"].value_counts()
print(f"Final assay distribution:\n{final_assay_counts}")

if len(final_assay_counts) > 1 or "SS2" not in final_assay_counts.index:
    print("⚠️  WARNING: Non-SS2 cells found in final dataset!")
else:
    print("✅ All cells are SS2 as expected!")

# Check for any obvious data corruption
print(f"X matrix - Type: {type(merged_adata.X)}, Format: {merged_adata.X.format}")
print(f"X matrix - Non-zero elements: {merged_adata.X.nnz:,}")
print(f"X matrix - Min value: {merged_adata.X.min()}")
print(f"X matrix - Max value: {merged_adata.X.max()}")

# Verify gene names are consistent
if not all(merged_adata.var_names == var_final.index):
    print("❌ ERROR: Gene names inconsistent in final object!")
else:
    print("✅ Gene names consistent in final object!")

# Check obs index integrity
if merged_adata.obs.index.duplicated().any():
    print("⚠️  WARNING: Duplicate indices found in obs!")
    print(f"Number of duplicates: {merged_adata.obs.index.duplicated().sum()}")
else:
    print("✅ No duplicate indices in obs!")

# %%
# Save the Final Merged Dataset Efficiently
output_path = "/gpfs/commons/datasets/controlled/CZI/tabula-sapiens/TabulaSapiens_v2/GeneExpressionMatrices/merged_tabula_sapiens.h5ad"
print(f"\nSaving to: {output_path}")
merged_adata.write_h5ad(output_path, compression="gzip")

# SANITY CHECK 12: Verify saved file
print("\n=== POST-SAVE VERIFICATION ===")
try:
    # Read back the saved file to verify it wasn't corrupted
    test_adata = sc.read_h5ad(output_path)
    print(f"✅ Successfully read back saved file")
    print(f"Saved file shape: {test_adata.shape}")
    
    # Quick verification that data matches
    if test_adata.shape == merged_adata.shape:
        print("✅ Saved file shape matches original")
    else:
        print("❌ ERROR: Saved file shape doesn't match!")
        
    # Check a few values to ensure no corruption
    if sp.issparse(test_adata.X) and sp.issparse(merged_adata.X):
        sample_match = (test_adata.X[:100, :100] != merged_adata.X[:100, :100]).nnz == 0
        if sample_match:
            print("✅ Sample data verification passed")
        else:
            print("⚠️  WARNING: Sample data doesn't match - possible corruption")
    
    del test_adata  # Free memory
    
except Exception as e:
    print(f"❌ ERROR reading saved file: {e}")

print(f"\n🎉 Successfully merged {processing_stats['datasets_processed']} datasets!")
print(f"Final shape: {merged_adata.shape}")
print(f"Total SS2 cells preserved: {merged_adata.n_obs:,}")

Current working directory: /gpfs/commons/home/kisaev/Leaflet-analysis/tabula_sapien


/gpfs/commons/home/kisaev/miniconda3/envs/LeafletSC/lib/python3.10/site-packages/cudf/utils/_ptxcompiler.py:64: UserWarning: Error getting driver and runtime versions:

stdout:



stderr:

Traceback (most recent call last):
  File "<string>", line 4, in <module>
  File "/gpfs/commons/home/kisaev/miniconda3/envs/LeafletSC/lib/python3.10/site-packages/numba/cuda/cudadrv/driver.py", line 295, in __getattr__
    raise CudaSupportError("Error at driver init: \n%s:" %
numba.cuda.cudadrv.error.CudaSupportError: Error at driver init: 

CUDA driver library cannot be found.
If you are sure that a CUDA driver is installed,
try setting environment variable NUMBA_CUDA_DRIVER
with the file path of the CUDA driver shared library.
:


Not patching Numba
  warnings.warn(msg, UserWarning)
/gpfs/commons/home/kisaev/miniconda3/envs/LeafletSC/lib/python3.10/site-packages/cudf/utils/gpu_utils.py:62: UserWarning: Failed to dlopen libcuda.so.1
  warnings.warn(str(e))


scanpy==1.10.4 anndata==0.11.3 umap==0.5.6 numpy==1.26.4 scipy==1.14.1 pandas==2.2.3 scikit-learn==1.6.1 statsmodels==0.14.5 igraph==0.11.5 louvain==0.8.2 pynndescent==0.5.12


  0%|          | 0/28 [00:00<?, ?it/s]

Reading /gpfs/commons/datasets/controlled/CZI/tabula-sapiens/TabulaSapiens_v2/GeneExpressionMatrices/Large_Intestine_TSP1_30_version2d_10X_smartseq_scvi_Nov122024.h5ad


...

100%|██████████| 28/28 [00:13<00:00,  2.04it/s]

After gene subsetting: 812 cells, 61806 genes
✅ Gene order consistent with previous datasets
✅ X is already sparse (csr)
Expression matrix - Shape: (812, 61806), Non-zero: 2,155,408
Sparsity: 95.71%

--- Processing dataset 27 ---
Initial: 3,055 cells, 61,806 genes
SS2 cells in this dataset: 0
❌ Skipping: No SS2 cells

=== PROCESSING SUMMARY ===
Datasets loaded: 28
Datasets processed: 25
Datasets skipped (no SS2): 3
Datasets skipped (no raw_counts): 0
Total SS2 cells kept: 41,501
Expected total from initial count: 41,501
✅ Cell count matches initial SS2 count!

=== MERGING DATA ===
Merging sparse matrices and metadata...
Expected cells from X matrices: 41,501
Expected cells from obs DataFrames: 41,501


Merged X shape: (41501, 61806)
Merged obs shape: (41501, 40)
var shape: (61806, 11)
✅ Merged dimensions are consistent!

=== FINAL VERIFICATION ===
Final AnnData shape: (41501, 61806)
Final cell count: 41,501
Final gene count: 61,806
Final assay distribution:
assay
SS2    41501
Name: count, dtype: int64
✅ All cells are SS2 as expected!
X matrix - Type: <class 'scipy.sparse._csr.csr_matrix'>, Format: csr
X matrix - Non-zero elements: 89,861,163


/gpfs/commons/home/kisaev/miniconda3/envs/LeafletSC/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


X matrix - Min value: 0
X matrix - Max value: 17453044
✅ Gene names consistent in final object!
✅ No duplicate indices in obs!

Saving to: /gpfs/commons/datasets/controlled/CZI/tabula-sapiens/TabulaSapiens_v2/GeneExpressionMatrices/merged_tabula_sapiens.h5ad

=== POST-SAVE VERIFICATION ===
✅ Successfully read back saved file
Saved file shape: (41501, 61806)
✅ Saved file shape matches original
✅ Sample data verification passed

🎉 Successfully merged 25 datasets!
Final shape: (41501, 61806)
Total SS2 cells preserved: 41,501
